In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'huggingface-hub>=0.26.0', 'python-dotenv>=1.0.0',
    'pyyaml>=6.0', 'requests>=2.32.0',
], check=True)

In [ ]:
import os, json, time, threading
from pathlib import Path
from datetime import datetime
import yaml, requests
from huggingface_hub import HfApi, CommitOperationAdd

WORK_DIR        = Path('/kaggle/working')
STREAMS_DIR     = WORK_DIR / 'streams'
MANIFEST_PATH   = WORK_DIR / 'e2e_episodes.jsonl'
CHECKPOINT_PATH = WORK_DIR / 'checkpoint_p5c.json'
CONFIG_DIR      = Path('/kaggle/input/datasets/mirza176528/s2s-pipline-v2-0-2/config')

WAVE_SIZE_BYTES = 500 * 1024 * 1024
BATCH_SIZE      = 50
MAX_RETRIES     = 12
COMMIT_DELAY    = 3.0

In [ ]:
def load_secrets():
    try:
        from kaggle_secrets import UserSecretsClient
        c = UserSecretsClient()
        s = {k: c.get_secret(k) for k in ['HF_TOKEN_PRIMARY','HF_TOKEN_SECONDARY','HF_TOKEN_TERTIARY']}
        print('[secrets] Kaggle'); return s
    except Exception: pass
    env_file = Path('.env')
    if env_file.exists():
        from dotenv import load_dotenv; load_dotenv(env_file)
    required = ['HF_TOKEN_PRIMARY','HF_TOKEN_SECONDARY','HF_TOKEN_TERTIARY']
    missing = [k for k in required if not os.environ.get(k)]
    if missing: raise RuntimeError(f'Missing: {missing}')
    return {k: os.environ[k] for k in required}

SECRETS       = load_secrets()
HF_TOKEN      = SECRETS['HF_TOKEN_PRIMARY']
HF_TOKEN_SEC  = SECRETS.get('HF_TOKEN_SECONDARY', HF_TOKEN)
with open(CONFIG_DIR / 'hf_repos.yaml') as f: repos_cfg = yaml.safe_load(f)
STAGE45_REPO = repos_cfg['repos']['stage45_e2e']['repo_id']
DOM_REPOS = {
    'restaurant': (repos_cfg['repos']['domain_restaurant']['repo_id'], HF_TOKEN_SEC),
    'banking':    (repos_cfg['repos']['domain_banking']['repo_id'],    HF_TOKEN_SEC),
    'healthcare': (repos_cfg['repos']['domain_healthcare']['repo_id'], HF_TOKEN_SEC),
    'education':  (repos_cfg['repos']['domain_education']['repo_id'],  HF_TOKEN_SEC),
}
HF_API = HfApi(token=HF_TOKEN)
print(f'[config] stage45: {STAGE45_REPO}')
for d, (r, _) in DOM_REPOS.items(): print(f'[config] {d}: {r}')

In [ ]:
def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        try:
            with open(CHECKPOINT_PATH) as f: state = json.load(f)
            print(f'[checkpoint] local — uploaded={state["stats"]["uploaded"]}')
            return state
        except Exception: pass
    print('[checkpoint] fresh start')
    return {
        'uploaded_ids': [], 'failed_ids': [],
        'manifest_uploaded': False, 'stats_uploaded': False,
        'stats': {'uploaded': 0, 'failed': 0, 'waves': 0},
        'last_updated': None,
    }

cp_lock = threading.Lock()

def save_checkpoint(state, upload=False):
    with cp_lock:
        state['last_updated'] = datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ')
        tmp = str(CHECKPOINT_PATH) + '.tmp'
        with open(tmp, 'w') as f: json.dump(state, f)
        os.replace(tmp, str(CHECKPOINT_PATH))
    if not upload: return
    for attempt in range(6):
        try:
            HF_API.upload_file(path_or_fileobj=json.dumps(state).encode(),
                path_in_repo='checkpoint_p5c.json', repo_id=STAGE45_REPO,
                repo_type='dataset', commit_message='p5c checkpoint')
            return
        except Exception: time.sleep(min(2**attempt, 60))

state        = load_checkpoint()
uploaded_set = set(state['uploaded_ids'])

In [ ]:
manifest_records = {}
if MANIFEST_PATH.exists():
    with open(MANIFEST_PATH, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rec = json.loads(line)
                manifest_records[rec['episode_id']] = rec

all_streams = sorted(STREAMS_DIR.glob('*.npy'))
pending     = [p for p in all_streams if p.stem not in uploaded_set]
total_gb    = sum(p.stat().st_size for p in pending) / 1024**3
print(f'[p5c] {len(all_streams)} streams, {len(uploaded_set)} uploaded, {len(pending)} pending ({total_gb:.2f} GB)')

def commit_batch_to_repo(batch, repo_id, token, wave_num, batch_idx, total):
    api = HfApi(token=token)
    ops = [CommitOperationAdd(path_in_repo=f'streams/{p.name}', path_or_fileobj=str(p))
           for p in batch if p.exists()]
    if not ops: return True
    for attempt in range(MAX_RETRIES):
        try:
            api.create_commit(repo_id=repo_id, repo_type='dataset',
                commit_message=f'streams wave {wave_num} batch {batch_idx}/{total}',
                operations=ops)
            time.sleep(COMMIT_DELAY)
            return True
        except Exception as e:
            if attempt == MAX_RETRIES - 1:
                print(f'  [commit] FAILED wave {wave_num} batch {batch_idx}: {e}')
                return False
            time.sleep(min(2**attempt, 120))
    return False

domain_buffers = {d: [] for d in DOM_REPOS}
domain_bytes   = {d: 0  for d in DOM_REPOS}
wave_nums      = {d: 1  for d in DOM_REPOS}

def flush_domain(domain, force=False):
    buf = domain_buffers[domain]
    if not buf: return
    if not force and domain_bytes[domain] < WAVE_SIZE_BYTES: return
    repo_id, token = DOM_REPOS[domain]
    wn      = wave_nums[domain]
    batches = [buf[i:i+BATCH_SIZE] for i in range(0, len(buf), BATCH_SIZE)]
    for idx, batch in enumerate(batches):
        ok = commit_batch_to_repo(batch, repo_id, token, wn, idx+1, len(batches))
        if ok:
            with cp_lock:
                for p in batch:
                    uploaded_set.add(p.stem)
                    state['uploaded_ids'].append(p.stem)
                state['stats']['uploaded'] += len(batch)
        else:
            with cp_lock:
                for p in batch: state['failed_ids'].append(p.stem)
                state['stats']['failed'] += len(batch)
    for p in buf: p.unlink(missing_ok=True)
    domain_buffers[domain] = []
    domain_bytes[domain]   = 0
    wave_nums[domain]     += 1
    state['stats']['waves'] += 1
    save_checkpoint(state, upload=True)
    print(f'  [{domain}] wave {wn} done')

for npy_path in pending:
    ep_id  = npy_path.stem
    rec    = manifest_records.get(ep_id, {})
    domain = rec.get('domain', 'restaurant')
    if domain not in domain_buffers:
        domain = 'restaurant'
    domain_buffers[domain].append(npy_path)
    domain_bytes[domain] += npy_path.stat().st_size
    flush_domain(domain)

for domain in DOM_REPOS:
    flush_domain(domain, force=True)

print('[p5c] all stream files uploaded to domain repos')

In [ ]:
if MANIFEST_PATH.exists() and not state['manifest_uploaded']:
    print('[manifest] uploading e2e_episodes.jsonl to stage45...')
    for attempt in range(MAX_RETRIES):
        try:
            HF_API.upload_file(
                path_or_fileobj=str(MANIFEST_PATH),
                path_in_repo='e2e_episodes.jsonl',
                repo_id=STAGE45_REPO,
                repo_type='dataset',
                commit_message='e2e_episodes.jsonl',
            )
            state['manifest_uploaded'] = True
            save_checkpoint(state, upload=False)
            print('[manifest] uploaded')
            break
        except Exception as e:
            time.sleep(min(2**attempt, 120))

total_recs = len(manifest_records)
domain_cnt = {}
diff_cnt   = {}
for rec in manifest_records.values():
    d = rec.get('domain','?'); domain_cnt[d] = domain_cnt.get(d,0)+1
    df = rec.get('difficulty','?'); diff_cnt[df] = diff_cnt.get(df,0)+1

stats = {
    'total_episodes': total_recs,
    'by_domain':      domain_cnt,
    'by_difficulty':  diff_cnt,
    'uploaded_streams': state['stats']['uploaded'],
    'failed_streams':   state['stats']['failed'],
    'generated_at':     datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ'),
}
for attempt in range(MAX_RETRIES):
    try:
        HF_API.upload_file(
            path_or_fileobj=json.dumps(stats, indent=2).encode(),
            path_in_repo='stats.json',
            repo_id=STAGE45_REPO,
            repo_type='dataset',
            commit_message='stats.json',
        )
        print('[stats] uploaded')
        break
    except Exception as e:
        time.sleep(min(2**attempt, 120))

print('\n[p5c] final summary')
print(f'  uploaded streams : {state["stats"]["uploaded"]}')
print(f'  failed streams   : {state["stats"]["failed"]}')
print(f'  domain repos     : {list(DOM_REPOS.keys())}')
print(f'\n[done] ALL PIPELINES COMPLETE')
print(f'[done] stage45 repo : https://huggingface.co/datasets/{STAGE45_REPO}')
for d, (r,_) in DOM_REPOS.items():
    print(f'[done] {d} repo : https://huggingface.co/datasets/{r}')